# Day 10, Steps 6-8: Blend, cold path, final system
This notebook **only loads saved models** (ALS from 05, both two-tower models from 06), so it runs in a few minutes with no training.

**Checks:** each check prints what I got next to what I expected (no asserts).

## Names used in this notebook
| Name | What it is |
|---|---|
| `train`, `val` | purchase tables (weeks 0-9, week 10) |
| `val_key`, `warm`, `cold` | validation answer key (one row per customer), split by `is_cold` |
| `build_customer_features(ids)` | same function as 06 (copied unchanged): customers -> indexed features |
| `article_inputs`, `row_to_article_id` | item tower inputs for the 38,603 train articles, and row -> article ID |
| `two_tower_recommend(user_tower, item_tower, customer_ids, k)` | top-k article IDs per customer (a 2D array) |
| `als_recommend(customer_ids, k)` | ALS top-k article IDs per **warm** customer (dot product, filter off) |
| `score(recommendations, bought_lists)` | recall@10 and NDCG@10 lists, with the Day 6 metric functions |
| `day9_user/item`, `day10_user/item` | the saved two-tower models (Day 9, Day 10 with ID dropout) |

**Recommend vs evaluate are separate:** the recommend functions return **lists of articles**; `score` turns lists into numbers. The blend needs lists, and the Day 16 API needs only recommend.

In [1]:
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

FEATURES_FOLDER = '../data/processed/features'
MODELS_FOLDER = '../models'

## Setup 1: Data and the Day 5 features

In [2]:
train = pd.read_parquet('../data/processed/train.parquet')
val = pd.read_parquet('../data/processed/val.parquet')
customers = pd.read_parquet('../data/RAW/customers.parquet')
customers_idx = pd.read_parquet('../data/processed/customer_join_idx.parquet')
articles = pd.read_parquet('../data/RAW/articles.parquet')
print('Train rows:', len(train), '| expected 2,638,103')
print('Val rows:  ', len(val), '| expected 230,980')


def load_vocab(name):
    table = pd.read_parquet(FEATURES_FOLDER + '/vocab_' + name + '.parquet')
    return dict(zip(table['value'], table['index']))

customer_vocab = load_vocab('customer')
article_vocab = load_vocab('article')
age_bucket_vocab = load_vocab('age_bucket')
club_member_status_vocab = load_vocab('club_member_status')
fashion_news_frequency_vocab = load_vocab('fashion_news_frequency')

with open(FEATURES_FOLDER + '/settings.json') as f:
    age_edges = json.load(f)['age_edges']
with open(FEATURES_FOLDER + '/price_scaling.json') as f:
    price_scaling = json.load(f)                 # the SAVED scaling from training, never recomputed
print('Customers:', len(customer_vocab), '| articles:', len(article_vocab), '| expected 444,431 and 38,603')
print('Price scaling:', price_scaling, '| expected transform log')

Train rows: 2638103 | expected 2,638,103
Val rows:   230980 | expected 230,980
Customers: 444431 | articles: 38603 | expected 444,431 and 38,603
Price scaling: {'price_log_mean': -3.9165074825286865, 'price_log_std': 0.7523115873336792, 'transform': 'log'} | expected transform log


## Setup 2: Customer features (the same function as 06, copied unchanged)
Copying a function is how training-serving skew starts, so Setup 6 proves this copy gives **exactly** the Day 9 and Day 10 validation numbers. (On Day 16, both notebooks and the API will import it from one `.py` file.)

In [3]:
customers_with_idx = customers.merge(customers_idx, on='customer_id', how='inner', validate='one_to_one')


def build_customer_features(customer_join_ids):
    # Any list of customers -> one row each, every feature as a vocabulary number. Cold -> customer_idx 0
    table = customers_with_idx[customers_with_idx['customer_join_idx'].isin(customer_join_ids)].copy()
    table['FN'] = table['FN'].fillna(0)
    table['Active'] = table['Active'].fillna(0)
    table['fashion_news_frequency'] = table['fashion_news_frequency'].replace('None', 'NONE')
    table['fashion_news_frequency'] = table['fashion_news_frequency'].map(fashion_news_frequency_vocab).fillna(0)
    table['club_member_status'] = table['club_member_status'].map(club_member_status_vocab).fillna(0)
    table['age'] = pd.cut(table['age'], bins=age_edges, right=False, labels=age_edges[:-1]).astype('float')
    table['age'] = table['age'].map(age_bucket_vocab).fillna(0)
    table['customer_idx'] = table['customer_join_idx'].map(customer_vocab).fillna(0)
    table = table.rename(columns={
        'age': 'age_bucket_idx',
        'club_member_status': 'club_member_status_idx',
        'fashion_news_frequency': 'fashion_news_frequency_idx',
    })
    return table[['customer_join_idx', 'customer_idx', 'age_bucket_idx', 'club_member_status_idx',
                  'fashion_news_frequency_idx', 'FN', 'Active']]

## Setup 3: Article inputs for the item tower
Price uses the **saved** mean and std from training (exactly what serving would do).

In [4]:
article_table = articles[articles['article_id'].isin(train['article_id'].unique())].copy()
article_table['article_idx'] = article_table['article_id'].map(article_vocab)
article_columns = ['product_code', 'product_type_no', 'product_group_name', 'graphical_appearance_name',
                   'colour_group_name', 'perceived_colour_master_name', 'department_no',
                   'index_group_name', 'section_name', 'garment_group_name']
for name in article_columns:
    article_table[name + '_idx'] = article_table[name].map(load_vocab(name)).fillna(0)

article_price = pd.read_parquet(FEATURES_FOLDER + '/article_median_price.parquet')
article_table = article_table.merge(article_price, on='article_id', how='left', validate='one_to_one')
article_table['price_scaled'] = ((np.log(article_table['median_price']) - price_scaling['price_log_mean'])
                                 / price_scaling['price_log_std'])

row_to_article_id = article_table['article_id'].to_numpy()       # row r of the article vectors -> article ID
print('Articles:', len(article_table), '| expected 38,603')
print('Scaled price min / max:', round(article_table['price_scaled'].min(), 2), '/',
      round(article_table['price_scaled'].max(), 2), '| expected about -4.28 / 4.30')

Articles: 38603 | expected 38,603
Scaled price min / max: -4.28 / 4.3 | expected about -4.28 / 4.30


## Setup 4: Load the saved models
The towers remember their own input names and dtypes (`tower.input`), so the key lists come **from the saved model**, not retyped.

In [5]:
day9_user = keras.models.load_model(MODELS_FOLDER + '/two_tower/user_tower.keras')
day9_item = keras.models.load_model(MODELS_FOLDER + '/two_tower/item_tower.keras')
day10_user = keras.models.load_model(MODELS_FOLDER + '/two_tower_iddrop/user_tower.keras')
day10_item = keras.models.load_model(MODELS_FOLDER + '/two_tower_iddrop/item_tower.keras')

als_user_factors = np.load(MODELS_FOLDER + '/als/user_factors.npy')
als_item_factors = np.load(MODELS_FOLDER + '/als/item_factors.npy')
print('ALS factors:', als_user_factors.shape, als_item_factors.shape, '| expected (444432, 64) (38604, 64)')


def tower_inputs(tower, table):
    # A feature table -> the tower's input dictionary, each column cast to the dtype the tower expects
    inputs = {}
    for name in tower.input:
        inputs[name] = table[name].to_numpy().astype(tower.input[name].dtype)
    return inputs

article_inputs = tower_inputs(day9_item, article_table)
print('User keys:', list(day9_user.input.keys()))
print('Item keys:', len(article_inputs), '| expected 12')

ALS factors: (444432, 64) (38604, 64) | expected (444432, 64) (38604, 64)
User keys: ['customer_idx', 'age_bucket_idx', 'club_member_status_idx', 'fashion_news_frequency_idx', 'FN', 'Active']
Item keys: 12 | expected 12


## Setup 5: Recommend, then score (separate jobs)

In [6]:
def two_tower_recommend(user_tower, item_tower, customer_ids, k):
    # Top-k article IDs for each customer, in the same order as customer_ids. Shape (customers, k)
    features = pd.DataFrame({'customer_join_idx': customer_ids}).merge(
        build_customer_features(customer_ids), on='customer_join_idx', how='left', validate='one_to_one')
    user_vectors = user_tower.predict(tower_inputs(user_tower, features), batch_size=4096, verbose=0)
    article_vectors = item_tower.predict(article_inputs, batch_size=4096, verbose=0)

    top_rows = []
    for start in range(0, len(user_vectors), 2000):                       # 2,000 customers at a time
        chunk_scores = user_vectors[start:start + 2000] @ article_vectors.T
        top_rows.append(tf.math.top_k(chunk_scores, k=k).indices.numpy())
    return row_to_article_id[np.vstack(top_rows)]


# ALS columns -> article IDs (column 0 = unknown, never a real article)
als_column_to_article_id = np.zeros(len(article_vocab) + 1, dtype='int64')
for article, column in article_vocab.items():
    als_column_to_article_id[column] = article


def als_recommend(customer_ids, k):
    # ALS top-k for WARM customers only (cold customers have no trained row). Plain dot product = filter off
    rows = pd.Series(customer_ids).map(customer_vocab).to_numpy()
    top_columns = []
    for start in range(0, len(rows), 2000):
        chunk_scores = als_user_factors[rows[start:start + 2000]] @ als_item_factors.T
        top_columns.append(tf.math.top_k(chunk_scores, k=k).indices.numpy())
    return als_column_to_article_id[np.vstack(top_columns)]


def recall_at_k(recommended, bought, k):
    hits = 0
    for item in recommended[:k]:
        if item in bought:
            hits = hits + 1
    return hits / len(bought)


def ndcg_at_k(recommended, bought, k):
    dcg = 0
    position = 0
    for item in recommended[:k]:
        position = position + 1
        if item in bought:
            dcg = dcg + 1 / math.log2(position + 1)
    ideal = 0
    for position in range(1, min(len(bought), k) + 1):
        ideal = ideal + 1 / math.log2(position + 1)
    return dcg / ideal


def score(recommendations, bought_lists):
    # Lists of recommended articles + what each customer bought -> recall@10 and NDCG@10 per customer
    recall_list = []
    ndcg_list = []
    for recs, bought in zip(recommendations, bought_lists):
        recall_list.append(recall_at_k(list(recs), set(bought), 10))
        ndcg_list.append(ndcg_at_k(list(recs), set(bought), 10))
    return np.array(recall_list), np.array(ndcg_list)

## Setup 6: Parity checks (prove this notebook matches 05 and 06)
If the copied feature code or the loading were different in any way, these numbers would not match.

In [7]:
val_key = val.groupby('customer_join_idx')['article_id'].unique().reset_index()
val_key['is_cold'] = ~val_key['customer_join_idx'].isin(train['customer_join_idx'].unique())
warm = val_key[~val_key['is_cold']].reset_index(drop=True)
cold = val_key[val_key['is_cold']].reset_index(drop=True)
print('Validation customers:', len(val_key), '| warm', len(warm), '| cold', len(cold), '| expected 72,019 / 48,507 / 23,512')

Validation customers: 72019 | warm 48507 | cold 23512 | expected 72,019 / 48,507 / 23,512


In [8]:
for title, user_tower, item_tower, expected in [
        ('Two-tower Day 9 ', day9_user, day9_item, 'all 0.0218  warm 0.0311  cold 0.0024'),
        ('Two-tower Day 10', day10_user, day10_item, 'all 0.0251  warm 0.0296  cold 0.0157')]:
    recall, ndcg = score(two_tower_recommend(user_tower, item_tower, val_key['customer_join_idx'], 10), val_key['article_id'])
    is_cold = val_key['is_cold'].to_numpy()
    print(title, '| all', round(recall.mean(), 4), ' warm', round(recall[~is_cold].mean(), 4),
          ' cold', round(recall[is_cold].mean(), 4), '| expected', expected)

als_recall, als_ndcg = score(als_recommend(warm['customer_join_idx'], 10), warm['article_id'])
print('ALS (filter off), warm | recall', round(als_recall.mean(), 4), ' NDCG', round(als_ndcg.mean(), 4), '| expected 0.0391 0.0273')

Two-tower Day 9  | all 0.0218  warm 0.0311  cold 0.0024 | expected all 0.0218  warm 0.0311  cold 0.0024
Two-tower Day 10 | all 0.0251  warm 0.0296  cold 0.0157 | expected all 0.0251  warm 0.0296  cold 0.0157
ALS (filter off), warm | recall 0.0391  NDCG 0.0273 | expected 0.0391 0.0273


In [9]:
# 6a-1: fill_up (copied from 05_baselines, Day 8 Step 1)
def fill_up(my_list, fallback, k=10):
    # Add fallback articles (skipping ones already in my_list) until the list has k articles
    filled = list(my_list)
    for article in fallback:
        if len(filled) == k:
            break
        if article not in filled:
            filled.append(article)
    return filled


In [10]:
# 6a-2: each customer's own purchases, newest first, at most 10 (copied from 05_baselines)
history = train.sort_values('t_dat', ascending=False)
history = history.drop_duplicates(subset=['customer_join_idx', 'article_id'], keep='first')
history = history.groupby('customer_join_idx').head(10)
past_purchases = history.groupby('customer_join_idx')['article_id'].apply(list).to_dict()
print('Customers with a history:', len(past_purchases), '| expected 444,431')


Customers with a history: 444431 | expected 444,431


In [11]:
# 6b: who has empty slots? Only they can show a difference between fillers
own_lengths = np.array([len(past_purchases[customer]) for customer in warm['customer_join_idx']])
has_empty = own_lengths < 10

print('Warm customers with empty slots:', has_empty.sum(), 'of', len(warm),
      '=', round(has_empty.mean() * 100, 1), '%')
print('Average empty slots (among them):', round((10 - own_lengths[has_empty]).mean(), 2))
print('Average own articles (all warm):  ', round(own_lengths.mean(), 2), '| expected 6.02 (Day 8)')


Warm customers with empty slots: 33681 of 48507 = 69.4 %
Average empty slots (among them): 5.74
Average own articles (all warm):   6.02 | expected 6.02 (Day 8)


In [12]:
# 6d: 30 candidates from each filler (30, so there are always enough NEW articles after skipping own ones)
warm_ids = warm['customer_join_idx']

last_week_top30 = train[train['week'] == 9]['article_id'].value_counts().head(30).index.to_list()
als_top30 = als_recommend(warm_ids, 30)
day9_top30 = two_tower_recommend(day9_user, day9_item, warm_ids, 30)
day10_top30 = two_tower_recommend(day10_user, day10_item, warm_ids, 30)

print('Popularity:', len(last_week_top30), 'articles (the same list for everyone)')
print('ALS:', als_top30.shape, '| Day 9:', day9_top30.shape, '| Day 10:', day10_top30.shape, '| expected (48507, 30) each')


Popularity: 30 articles (the same list for everyone)
ALS: (48507, 30) | Day 9: (48507, 30) | Day 10: (48507, 30) | expected (48507, 30) each


In [13]:
# 6e: blend = own purchases first, then the filler; score all warm and only the empty-slot customers
blend_results = []

def blend_and_score(filler_name, filler_lists):
    blended = []
    for customer, candidates in zip(warm['customer_join_idx'], filler_lists):
        blended.append(fill_up(past_purchases[customer], candidates))

    lengths = [len(recs) for recs in blended]
    print(filler_name.ljust(22), '| list lengths', min(lengths), 'to', max(lengths), '| expected 10 to 10')

    recall, ndcg = score(blended, warm['article_id'])
    blend_results.append({
        'Filler': filler_name,
        'Recall (all warm)': round(recall.mean(), 4),
        'NDCG (all warm)': round(ndcg.mean(), 4),
        'Recall (empty slots)': round(recall[has_empty].mean(), 4),
        'NDCG (empty slots)': round(ndcg[has_empty].mean(), 4),
    })

# Popularity is ONE list, so give every customer the same one
blend_and_score('A. Last-week popularity', [last_week_top30] * len(warm))
blend_and_score('B. ALS (filter off)', als_top30)
blend_and_score('C1. Two-tower Day 9', day9_top30)
blend_and_score('C2. Two-tower Day 10', day10_top30)


A. Last-week popularity | list lengths 10 to 10 | expected 10 to 10
B. ALS (filter off)    | list lengths 10 to 10 | expected 10 to 10
C1. Two-tower Day 9    | list lengths 10 to 10 | expected 10 to 10
C2. Two-tower Day 10   | list lengths 10 to 10 | expected 10 to 10


In [14]:
# 6f: the comparison. Filler A must reproduce the Day 8 repurchase baseline: 0.0584 / 0.0457
blend_table = pd.DataFrame(blend_results)
blend_table


,Filler,Recall (all warm),NDCG (all warm),Recall (empty slots),NDCG (empty slots)
0,A. Last-week popularity,0.0584,0.0457,0.0556,0.0436
1,B. ALS (filter off),0.0631,0.0478,0.0624,0.0467
2,C1. Two-tower Day 9,0.0592,0.0461,0.0567,0.0443
3,C2. Two-tower Day 10,0.0590,0.0460,0.0564,0.0440


In [15]:
# 7a: week 9 purchases, with each BUYER's age bucket (same feature function as everywhere else)
week9 = train[train['week'] == 9][['customer_join_idx', 'article_id']]
week9_buyers = build_customer_features(week9['customer_join_idx'].unique())[['customer_join_idx', 'age_bucket_idx']]
week9 = week9.merge(week9_buyers, on='customer_join_idx', how='left', validate='many_to_one')

print('Week 9 purchases:', len(week9), '| expected 240,845')
print('Missing age buckets:', week9['age_bucket_idx'].isna().sum(), '| expected 0')
print(week9['age_bucket_idx'].value_counts().sort_index())


Week 9 purchases: 240845 | expected 240,845
Missing age buckets: 0 | expected 0
age_bucket_idx
0.0      916
1.0     8755
2.0    48748
3.0    52734
4.0    45629
5.0    35931
6.0    36814
7.0    11318
Name: count, dtype: int64


In [16]:
# 7b: the top 10 articles inside each age bucket (bucket 0 = age unknown -> uses the global list instead)
global_top10 = week9['article_id'].value_counts().head(10).index.to_list()

bucket_top10 = {}
for bucket in sorted(week9['age_bucket_idx'].unique()):
    if bucket == 0:
        continue
    bucket_rows = week9[week9['age_bucket_idx'] == bucket]
    bucket_top10[bucket] = bucket_rows['article_id'].value_counts().head(10).index.to_list()

for bucket in bucket_top10:
    in_global = sum(1 for article in bucket_top10[bucket] if article in global_top10)
    print('Bucket', int(bucket), '| ages from', age_edges[int(bucket) - 1], '|', in_global, 'of its top 10 are in the global top 10')


Bucket 1 | ages from 16 | 3 of its top 10 are in the global top 10
Bucket 2 | ages from 20 | 6 of its top 10 are in the global top 10
Bucket 3 | ages from 25 | 6 of its top 10 are in the global top 10
Bucket 4 | ages from 30 | 6 of its top 10 are in the global top 10
Bucket 5 | ages from 40 | 5 of its top 10 are in the global top 10
Bucket 6 | ages from 50 | 5 of its top 10 are in the global top 10
Bucket 7 | ages from 60 | 5 of its top 10 are in the global top 10


In [17]:
# 7c: every cold customer gets their bucket's list (filled up from the global list if needed)
cold_features = pd.DataFrame({'customer_join_idx': cold['customer_join_idx']}).merge(
    build_customer_features(cold['customer_join_idx']), on='customer_join_idx', how='left', validate='one_to_one')

age_bucket_lists = []
for bucket in cold_features['age_bucket_idx']:
    own_bucket_list = bucket_top10.get(bucket, [])          # unknown age (bucket 0) -> empty -> all global
    age_bucket_lists.append(fill_up(own_bucket_list, global_top10))

lengths = [len(recs) for recs in age_bucket_lists]
print('List lengths:', min(lengths), 'to', max(lengths), '| expected 10 to 10')
print('Cold customers with unknown age (bucket 0):', (cold_features['age_bucket_idx'] == 0).sum())



List lengths: 10 to 10 | expected 10 to 10
Cold customers with unknown age (bucket 0): 148


In [18]:
# 7d: the three cold candidates, scored on the same 23,512 cold customers
cold_results = []

global_recall, global_ndcg = score([global_top10] * len(cold), cold['article_id'])
cold_results.append({'Cold path': '1. Last-week global popularity',
                     'Recall@10': round(global_recall.mean(), 4), 'NDCG@10': round(global_ndcg.mean(), 4)})

bucket_recall, bucket_ndcg = score(age_bucket_lists, cold['article_id'])
cold_results.append({'Cold path': '2. Last-week popularity per age bucket',
                     'Recall@10': round(bucket_recall.mean(), 4), 'NDCG@10': round(bucket_ndcg.mean(), 4)})

tt_recall, tt_ndcg = score(two_tower_recommend(day10_user, day10_item, cold['customer_join_idx'], 10), cold['article_id'])
cold_results.append({'Cold path': '3. Two-tower Day 10 (ID dropout)',
                     'Recall@10': round(tt_recall.mean(), 4), 'NDCG@10': round(tt_ndcg.mean(), 4)})

cold_table = pd.DataFrame(cold_results)
cold_table['vs global %'] = ((cold_table['Recall@10'] - cold_table['Recall@10'][0]) / cold_table['Recall@10'][0] * 100).round()
print('Expected: global 0.0194 (Day 6), two-tower 0.0157 (Day 10)')
cold_table


Expected: global 0.0194 (Day 6), two-tower 0.0157 (Day 10)


,Cold path,Recall@10,NDCG@10,vs global %
0,1. Last-week global popularity,0.0194,0.0124,0.0
1,2. Last-week popularity per age bucket,0.0216,0.0141,11.0
2,3. Two-tower Day 10 (ID dropout),0.0157,0.0107,-19.0


**Decision rule (b), Step 6:** ALS is the best filler (warm 0.0631 vs popularity 0.0584, +8%; on the 33,681 customers with empty slots 0.0624 vs 0.0556, +12%). Both two-tower models barely beat popularity (0.0592 / 0.0590) and lose clearly to ALS, so **rule (b) is not met**.

**Cold path, Step 7:** last-week popularity **per age bucket** wins (0.0216, +11% over global popularity's 0.0194; the two-tower model is 0.0157, −19%). So the cold features **do** carry signal: age alone changes what people buy (bucket 1, ages 16-19, shares only 3 of its top 10 with the global list). The network had that signal available but didn't use it well, probably because the customer ID dominates training and the five cold features are mixed through layers tuned mostly for warm customers; a direct "popular in your age group" list captures it more reliably. **Rule (a) is not met** either: two-tower cold is below both popularity lists.

## Step 8: Freeze the final system
Decided **today**, before opening the test week: if I chose the system after seeing test numbers, the test score would no longer be an honest estimate (the same rule as the ticket router's test day).

In [19]:
# Step 8: the FINAL system's validation numbers, measured end to end
# Warm: repurchase first, then ALS fills the empty slots.  Cold: last-week popularity per age bucket.
final_warm_lists = []
for customer, candidates in zip(warm['customer_join_idx'], als_top30):
    final_warm_lists.append(fill_up(past_purchases[customer], candidates))

final_warm_recall, final_warm_ndcg = score(final_warm_lists, warm['article_id'])
final_cold_recall, final_cold_ndcg = score(age_bucket_lists, cold['article_id'])

# "All" = every customer counted once (warm and cold arrays joined), then one average
final_all_recall = np.concatenate([final_warm_recall, final_cold_recall])
final_all_ndcg = np.concatenate([final_warm_ndcg, final_cold_ndcg])

print('Customers scored:', len(final_all_recall), '| expected 72,019')
print('FINAL  all  recall@10 =', round(final_all_recall.mean(), 4), '  NDCG@10 =', round(final_all_ndcg.mean(), 4))
print('FINAL  warm recall@10 =', round(final_warm_recall.mean(), 4), '  NDCG@10 =', round(final_warm_ndcg.mean(), 4), '| expected 0.0631 / 0.0478 (Step 6, filler B)')
print('FINAL  cold recall@10 =', round(final_cold_recall.mean(), 4), '  NDCG@10 =', round(final_cold_ndcg.mean(), 4), '| expected 0.0216 / 0.0141 (Step 7)')
print('Last-week popularity, all: 0.0184 / 0.0118')

Customers scored: 72019 | expected 72,019
FINAL  all  recall@10 = 0.0495   NDCG@10 = 0.0368
FINAL  warm recall@10 = 0.0631   NDCG@10 = 0.0478 | expected 0.0631 / 0.0478 (Step 6, filler B)
FINAL  cold recall@10 = 0.0216   NDCG@10 = 0.0141 | expected 0.0216 / 0.0141 (Step 7)
Last-week popularity, all: 0.0184 / 0.0118


### The frozen system (run once on the test week tomorrow, no changes)

| Part | What it does |
|---|---|
| **Warm path** (customer bought something in train) | their own purchases first, newest first, each once, at most 10; then **ALS (filter off)** fills the remaining slots, skipping articles already in the list (30 ALS candidates, so every list has exactly 10) |
| **Cold path** (no train history) | **last-week popularity in the customer's age bucket** (week 9 purchases of buyers in the same bucket); unknown age, or a bucket with fewer than 10 articles, is filled up from the global last-week top 10 |

**Files used:**
- `models/als/user_factors.npy`, `item_factors.npy` (Day 7 ALS)
- `data/processed/train.parquet` (own purchases, week 9 popularity)
- `data/processed/features/vocab_*.parquet`, `settings.json` (customer and article mappings, age edges)
- `data/RAW/customers.parquet` + `data/processed/customer_join_idx.parquet` (age bucket via `build_customer_features`)

**Validation numbers of the final system** (all 72,019 validation customers; tomorrow's test result is compared with these):

| | Recall@10 | NDCG@10 |
|---|---|---|
| **All** | **0.0495** | **0.0368** |
| Warm (48,507) | 0.0631 | 0.0478 |
| Cold (23,512) | 0.0216 | 0.0141 |
| *Last-week popularity, all* | *0.0184* | *0.0118* |

That's **+169%** over last-week popularity and **+8%** over the repurchase baseline (0.0457).

**Not used, and why:**
- **Two-tower Day 9:** as filler it barely beat popularity (0.0592) and lost to ALS (0.0631); cold 0.0024 (row 0 never trained).
- **Two-tower Day 10 (ID dropout):** filler 0.0590, below ALS; cold 0.0157, below both global (0.0194) and age-bucket (0.0216) popularity. It met neither decision rule, so it is not served.
- **ALS alone:** 0.0391 warm, below repurchase-first blends; can't serve cold customers.
- **Global popularity for cold:** 0.0194, beaten by the age-bucket version.

**Known limitations:**
- The two-tower models aren't fully trained (Day 10 capped at 8 epochs and still rising), so "not served" is a verdict on *these* runs, not on two-tower models in general.
- **Cold articles can't be recommended:** the 1,225 articles that are new in validation have no ALS column, and none of the lists include them.
- Popularity and the age-bucket lists use week 9 only; on the test week they'll be one week older than on validation (a small, known staleness).

## Day 10 interview question
*"A third of your customers had no purchase history. Walk me through how your model failed them, how you found out why, and what you finally served them."*

> About a third of my validation customers were new, so they all shared row 0 of the customer embedding, the "unknown" row. While training the two-tower model I tracked recall separately for new customers, and it got *worse* every epoch, from 0.011 to 0.002. The reason: row 0 never appears in training, so its gradient is always zero and it stays random, and as the model leans more on the customer ID, new customers inherit that random direction. I fixed it with ID dropout: during training I set 20% of customer IDs to 0, so row 0 learns to mean "a customer I only know by their features". That lifted new-customer recall 6.5 times, to 0.016, but it was still below plain last-week popularity at 0.019. So I tried a one-line baseline, last-week bestsellers within the customer's age group, and it reached 0.022, 11% above popularity. That proved the features do carry signal; my hypothesis is that the network under-used them because the customer ID dominates training, and the test would be a user tower with no ID at all. What I shipped: age-group popularity for new customers, and for returning customers their own recent purchases topped up with ALS: 0.0495 recall overall, 2.7 times popularity.